In [ ]:
!pip install -q "ultralytics==8.4.115"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import json
import platform
import sys
import torch
import ultralytics

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Grounded-PPE-Safety-Copilot/phase1_outputs"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

environment = {
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "ultralytics": ultralytics.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
}

print(json.dumps(environment, indent=2, ensure_ascii=False))

with open(OUTPUT_DIR / "environment.json", "w") as file:
    json.dump(environment, file, indent=2, ensure_ascii=False)

assert torch.cuda.is_available(), "请先为 Colab 启用 GPU"

In [ ]:
from os import name
from google.colab import files

uploaded = files.upload()
image_paths = [Path("/content") / name for name in uploaded] # good for using for other times

print("uploaded number: ", len(image_paths))
for path in image_paths:
  print(path)

In [ ]:
from ultralytics import YOLOWorld

MODEL_NAME = "yolov8s-worldv2.pt"
PROMPTS = ["person", "hard hat", "safety vest"]

model = YOLOWorld(MODEL_NAME)
model.set_classes(PROMPTS)

all_results = []

for image_path in image_paths:
    result = model.predict(
        source=str(image_path),
        conf=0.20,
        iou=0.50,
        imgsz=640,
        device=0,
        verbose=False,
    )[0]

    stem = image_path.stem

    result.save(
        filename=str(OUTPUT_DIR / f"{stem}_prediction.jpg")
    )

    json_path = OUTPUT_DIR / f"{stem}_detections.json"
    json_path.write_text(
        result.to_json(),
        encoding="utf-8",
    )

    all_results.append({
        "image": image_path.name,
        "detections": len(result.boxes),
        "speed_ms": result.speed,
    })

print(json.dumps(all_results, indent=2, ensure_ascii=False))

模型输入包括一张 RGB 图片和一组文本类别提示词，本实验使用 person、hard hat 和 safety vest。模型对每个预测目标输出边界框坐标、类别编号、类别名称和置信度。边界框采用 xyxy 格式，分别表示左上角和右下角坐标。降低置信度阈值通常能够发现更多目标并提高召回率，但也可能增加误检；提高阈值通常能够减少误检，但可能增加漏检。YOLO‑World 能根据文本提示进行零样本检测，但在小目标、遮挡、多人场景和领域外图片上可能出现漏检或错误类别。它只能检测目标，不能可靠判断某个头盔或背心属于哪名人员，该问题将在后续人员–PPE 关联阶段解决。

In [ ]:
from collections import Counter
from IPython.display import display
from PIL import Image
import json

baseline_summary = []

for image_path in image_paths:
    prediction_path = OUTPUT_DIR / f"{image_path.stem}_prediction.jpg"
    json_path = OUTPUT_DIR / f"{image_path.stem}_detections.json"

    print(f"\n==={image_path.name}===")
    display(Image.open(prediction_path))

    detections = json.loads(json_path.read_text(encoding="utf-8"))
    class_counts = Counter(
        detection.get("name", "unknown") for detection in detections
    )

    confidences = [
        float(detection.get("confidence", 0))
        for detection in detections
    ]

    summary = {
        "image": image_path.name,
        "total_detections": len(detections),
        "class_counts": dict(class_counts),
        "average_confidence": (
            sum(confidences) / len(confidences)
            if confidences else 0
        ),
    }

    baseline_summary.append(summary)
    print(json.dumps(summary, indent=2, ensure_ascii=False))

summary_path = OUTPUT_DIR / "baseline_summary.json"
summary_path.write_text(
    json.dumps(baseline_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(f"\n结果已保存到：{summary_path}")